# Operate: 프로덕션에서 Managed Agents 운영하기

다른 Managed Agents 쿡북 대부분은 에이전트 루프 자체, 즉 픽스처를 상대로 유용한 일을 하게 만드는 데 집중합니다. 이 노트북은 그 루프를 둘러싼 기계 장치, 즉 Managed Agents 앱을 실제 사용자 앞에 내놓기 전에 필요한 부품들을 다룹니다.

1. 모든 호출을 애플리케이션으로 왕복시키지 않고 에이전트가 SaaS API와 직접 대화해야 할 때, 커스텀 도구 대신 쓰는 **MCP 툴셋**.
2. 최종 사용자별 자격 증명을 담는 **볼트(vault)**. 각 사용자의 GitHub / Linear / Slack 토큰이 서로 분리되고 감사 기록이 깔끔해집니다.
3. 오래 유지되는 HTTP 연결을 열어 두지 않고 사람 개입 작업을 구동하는 **웹훅**.
4. 시간이 지나며 워크스페이스에 쌓이는 것들을 관리하는 **리소스 수명 주기** 동사(list, retrieve, update, archive, delete).

네 가지를 모두 거치는 종단 간 흐름을 하나 만들어 보겠습니다. 가상의 최종 사용자를 위한 볼트를 만들고, GitHub MCP 자격 증명을 붙이고, 그 자격 증명을 서버 측에서 사용하는 에이전트 세션을 실행하고, 실제 프로덕션 서버에서 같은 세션을 구동하려면 등록해야 할 웹훅 핸들러를 보여 준 뒤, 나중에 정리할 때 쓰는 관리 동사들을 살펴봅니다.

이 노트북에는 환경에 `GITHUB_TOKEN`이 필요합니다.

In [ ]:
import os

from anthropic import Anthropic
from utilities import stream_until_end_turn, wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = Anthropic()

GH_TOKEN = os.environ.get("GITHUB_TOKEN")
if not GH_TOKEN:
    raise SystemExit("Set GITHUB_TOKEN to run this notebook.")

## 개념: MCP 툴셋과 볼트

**MCP 툴셋**은 커스텀 도구(gate 노트북)와 `resources=` 마운트(orchestrate 노트북)에 이은 세 번째 확장 패턴입니다. MCP 툴셋은 [Model Context Protocol](https://modelcontextprotocol.io/)을 구현한 외부 서버를 에이전트에 연결해 줍니다. 에이전트는 애플리케이션을 왕복하지 않고 샌드박스 안에서 그 서버의 도구를 직접 호출합니다. Anthropic이 호출을 중계하고, 서버가 응답하고, 에이전트는 계속 나아갑니다. 공개 SaaS API의 대다수(GitHub, Slack, Linear, Stripe, Notion, Salesforce, Asana 등)는 이미 MCP 서버가 있거나 반나절이면 감쌀 수 있고, 어느 쪽이든 좋은 MCP 후보입니다.

경험칙: 서비스가 공개 인터넷에서 베어러 토큰으로 접근 가능하다면 MCP 툴셋이 통합니다. 여러분의 네트워크 안에서만 접근 가능하다면 대신 커스텀 도구를 쓰세요. gate 노트북이 그것을 다룹니다.

**볼트**는 "토큰을 어디에 두지?"라는 질문에 대한 답입니다. 세션 생성 시점에 토큰 하나를 하드코딩하는 방식은 단일 테넌트 구성에서는 통하지만, 최종 사용자가 생기는 순간 무너집니다. 사용자마다 자기 GitHub 자격 증명이 필요하고, 서로 격리해 두어야 합니다. 볼트는 사용자별 자격 증명 컨테이너로, 한 번 등록해 두고 그 사용자를 위해 만드는 모든 세션에서 ID로 참조합니다. 자체 비밀 저장소를 운영할 필요도, 매 요청에 토큰을 실어 보낼 필요도 없고, 감사 기록이 볼트에 묶여 있어 에이전트가 어떤 최종 사용자를 대신해 동작했는지 항상 알 수 있습니다.

## 1. 최종 사용자를 위한 볼트 만들기

볼트에는 콘솔에 표시되는 `display_name`과, 보통 내부 사용자 ID를 담아 두는 `metadata` 딕셔너리가 있습니다. 이를 통해 볼트를 여러분의 데이터베이스 레코드와 연결할 수 있습니다.

In [ ]:
vault = client.beta.vaults.create(
    display_name="Cookbook demo user",
    metadata={"internal_user_id": "u_demo_001", "team": "engineering"},
)
print(f"vault: {vault.id}")

## 2. MCP 자격 증명 붙이기

자격 증명은 볼트 아래에 놓입니다. 각 자격 증명은 MCP 서버 URL과, 에이전트가 그 서버를 호출할 때 사용할 토큰을 짝지어 둡니다. GitHub Copilot MCP 서버라면 정적 베어러 토큰(여러분의 GitHub PAT)이 가장 단순한 형태입니다. API는 필요한 서비스를 위해 갱신을 포함한 완전한 OAuth 흐름도 지원하며, 두 형태 모두 `auth=`로 처리합니다.

In [ ]:
credential = client.beta.vaults.credentials.create(
    vault_id=vault.id,
    display_name="GitHub Copilot",
    auth={
        "type": "static_bearer",
        "mcp_server_url": "https://api.githubcopilot.com/mcp/",
        "token": GH_TOKEN,
    },
)
print(f"credential: {credential.id}")

## 3. 세션에서 볼트 참조하기

`sessions.create`에 `vault_ids=[vault.id]`를 넘기면 API가 도구를 호출할 때마다 해당 MCP 서버 URL을 찾아냅니다. 에이전트는 토큰 자체를 보지 못하고, 여러분도 요청에 토큰을 실어 보낼 필요가 없습니다. 에이전트 정의는 평소처럼 MCP 서버를 나열하기만 하고, 자격 증명 연결은 세션 생성 시점에 이뤄집니다.

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-operate",
    model=MODEL,
    system="You navigate GitHub repositories on behalf of the logged-in user.",
    mcp_servers=[
        {
            "type": "url",
            "name": "github",
            "url": "https://api.githubcopilot.com/mcp/",
        }
    ],
    tools=[
        {
            "type": "mcp_toolset",
            "mcp_server_name": "github",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-operate-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    vault_ids=[vault.id],
    title="Operate demo",
)
print(f"session: {session.id}")

## 4. 최종 사용자로서 한 턴 실행하기

이제 에이전트가 GitHub를 상대로 하는 모든 일이 볼트의 자격 증명을 거칩니다. 여러분의 시스템에서 감사하기도 간단합니다. 1단계에서 설정한 메타데이터로 볼트가 사용자와 묶여 있으므로, 어떤 최종 사용자를 대신한 동작인지 정확히 알 수 있습니다.

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "List the three most recently updated repos in the anthropics org.",
                }
            ],
        }
    ],
)
print("--- vault-backed MCP call ---")
stream_until_end_turn(client, session.id)

## 5. 프로덕션 사람 개입을 위한 웹훅

gate 노트북의 스트리밍 패턴은 모든 것이 한 프로세스에서 일어나므로 개발 중에는 편리하지만, 사람이 검토하는 동안 HTTP 연결을 열어 둡니다. 확장성이 없고 프로세스 재시작도 견디지 못합니다. 프로덕션 패턴에서는 대신 콘솔에 `session.status_idled`에 반응하는 웹훅을 등록합니다. 이 이벤트는 에이전트가 작업을 마쳤거나 도구 결과를 기다리고 있다는 신호입니다.

웹훅이 발생하면 여러분의 서버는 이벤트를 살펴보고, 대기 중인 에스컬레이션을 검토자 앞에 올려 두고, 사람이 끝내는 시점에 `user.custom_tool_result`를 POST로 돌려보냅니다. 세션은 여러분이 응답할 때까지 그저 유휴 상태로 있으며, 여러분 쪽에 오래 유지되는 연결이 없습니다.

웹훅 등록은 콘솔의 **Settings → Webhooks**에서 한 번만 하면 됩니다. 생성 시 한 번만 표시되는 `whsec_...` 서명 시크릿을 받게 되니 비밀 관리 시스템에 보관하세요.

**아래 블록은 노트북 셀이 아니라 참고 구현입니다.** 여러분의 서버로 복사해 쓰세요. 이 쿡북이 설치하지 않는 FastAPI에 의존하며, 이 노트북 흐름의 일부로 실행되지 않습니다. gate 노트북의 에이전트 정의와 짝지으면 프로덕션 서버에서 gate 워크플로를 처음부터 끝까지 구동하기에 충분합니다.

```python
import hmac
import hashlib
import json

from fastapi import FastAPI, Header, HTTPException, Request

app = FastAPI()
WEBHOOK_SECRET = "whsec_..."  # from Console, load from your secrets manager


def verify(body: bytes, sig: str) -> bool:
    expected = hmac.new(WEBHOOK_SECRET.encode(), body, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, sig)


@app.post("/webhooks/anthropic")
async def receive(req: Request, x_anthropic_signature: str = Header()):
    body = await req.body()
    if not verify(body, x_anthropic_signature):
        raise HTTPException(401)

    event = json.loads(body)
    session_id = event["resource_id"]

    if event["event_type"] == "session.status_idled":
        # Agent went idle. Either it's done OR it called escalate()
        # and is waiting on a user.custom_tool_result. Look at the
        # latest events to decide.
        events = client.beta.sessions.events.list(session_id=session_id)
        pending = [
            e for e in events.data
            if e.type == "agent.custom_tool_use" and e.name == "escalate"
            and not has_result(events, e.id)
        ]
        if pending:
            for tu in pending:
                enqueue_for_review(session_id, tu.id, tu.input)
        else:
            finalize_run(session_id)

    return {"ok": True}


# Called from your review UI once the human decides
def submit_review(session_id: str, custom_tool_use_id: str, decision: str):
    client.beta.sessions.events.send(
        session_id=session_id,
        events=[{
            "type": "user.custom_tool_result",
            "custom_tool_use_id": custom_tool_use_id,
            "content": [{"type": "text",
                         "text": json.dumps({"human_decision": decision})}],
        }],
    )
```

에이전트에 응답하는 코드는 gate 노트북의 A부 루프와 동일합니다. 달라지는 것은 여러분의 서버가 할 일이 생겼다는 사실을 어떻게 알게 되느냐뿐입니다. 로컬 루프가 이벤트를 당겨 오는 대신, 웹훅이 여러분의 일정에 맞춰 알림을 밀어 줍니다.

## 6. 리소스 수명 주기: list, retrieve, update, archive

API의 모든 리소스, 즉 에이전트, 환경, 세션, 볼트, 자격 증명은 같은 다섯 동사 패턴을 노출합니다. `list`, `retrieve`, `update`, `archive`, 그리고 (일부는) `delete`입니다. 에이전트를 대상으로 전체 세트를 시연한 뒤, 다른 리소스마다 사용할 수 있는 동사를 빠른 참고용으로 정리하겠습니다.

**archive와 delete의 차이:** `archive`는 감사와 조회를 위해 레코드를 남겨 두되, 실행 중인 컨테이너를 정리하고 해당 리소스가 워크스페이스 할당량에 잡히지 않게 합니다. `delete`는 레코드를 완전히 제거합니다. 대부분의 워크플로에서는 `archive`가 맞습니다. `delete`는 레코드를 확실히 없애야 할 때(예: 테스트 정리)만 쓰세요.

In [ ]:
listed = client.beta.agents.list(limit=5)
print(f"workspace has at least {len(listed.data)} agents")

retrieved = client.beta.agents.retrieve(agent.id)
print(f"retrieved agent: name={retrieved.name} version={retrieved.version}")

# Updating an agent produces a new version. Pass the current
# version to confirm you're updating from a known state, a
# concurrent update from another process will reject yours rather
# than silently overwriting.
updated = client.beta.agents.update(
    agent.id,
    version=agent.version,
    system="You navigate GitHub repositories. Be terse.",
)
print(f"updated to version {updated.version}")

# Every historical version is queryable.
versions = client.beta.agents.versions.list(agent_id=agent.id)
print(f"agent has {len(versions.data)} versions")

## 7. 정리

자격 증명과 볼트에는 각자의 아카이브 엔드포인트가 있습니다. 볼트를 아카이브해도 그 자격 증명이 자동으로 아카이브되지는 않으므로, 깔끔하게 정리하려면 자격 증명을 먼저 처리하세요.

In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.vaults.credentials.archive(credential.id, vault_id=vault.id)
client.beta.vaults.archive(vault.id)
print("archived")

## 다른 쿡북들

이 노트북은 프로덕션 관점의 마무리 편입니다. 아직 실행해 보지 않았다면 이 노트북이 감싸고 있는 워크플로 노트북들을 먼저 돌려 볼 만합니다.

- [`CMA_iterate_fix_failing_tests.ipynb`](CMA_iterate_fix_failing_tests.ipynb) — 출발점 노트북. 실패하는 테스트 스위트를 대상으로 실행-관찰-수정 루프를 돌며 에이전트, 환경, 세션, 파일 마운트, 스트리밍 이벤트 루프를 소개합니다.
- [`CMA_orchestrate_issue_to_pr.ipynb`](CMA_orchestrate_issue_to_pr.ipynb) — 모의 gh CLI로 이슈를 병합된 PR까지 끌고 가는 멀티턴 에이전트. 도중에 CI 실패와 리뷰 코멘트에서 회복합니다.
- [`CMA_explore_unfamiliar_codebase.ipynb`](CMA_explore_unfamiliar_codebase.ipynb) — 낡은 문서 함정이 심어진 근거 확보 패턴. 실행 중인 세션에 컨텍스트를 더 넣는 `sessions.resources.add`도 함께 보여 줍니다.
- [`CMA_gate_human_in_the_loop.ipynb`](CMA_gate_human_in_the_loop.ipynb) — 사람 개입 워크플로를 위한 커스텀 도구 `decide()`와 `escalate()` 왕복. 위의 웹훅 참고 블록과 짝을 이룹니다.